# 02 · The linear base — enetreg2 → linbest (the locked floor)

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first.** Each base-form QLIKE is a cluster walk-forward `resid_amortized` base fit (full-OOS,
deferred Duan smearing). **The prediction cache is not in this repo**, so every number is shown **with its
exact reproduce-command** — base-alone QLIKE is the `base_alone_qlike=…` line a `prep` build prints; the
block attribution is the one `fwl_attribution.py` run that reproduces 0.12314. We fold the locked-base
feature machinery (`_cumrv_close`, `_regime_interactions`) and the FWL block partition (`block_of`) from
source — no bare pasted number.

In [ ]:
import html, inspect, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", None)

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))

# Collapsible, theme-following source display (one <details> per function; folded) — the same
# helper as notebooks/results/bucket_sweep.ipynb.
def _details(f, open_=False):
    path = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(path + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f): return Markdown(_details(f))   # thin: just this function (no transitive tree)

from resid_amortized import _cumrv_close, _regime_interactions, CACHE_ROOT
from fwl_attribution import block_of   # the exact block partition the FWL attribution uses

# "No bare number" invariant: every quantitative row must carry a non-empty reproduce-command.
def assert_backed(df, col="reproduce"):
    miss = df[df[col].astype(str).str.strip() == ""]
    assert miss.empty, f"unbacked numbers (no reproduce-command): {miss.index.tolist()}"
    return len(df)

PY = "$PY"  # cluster python (conda env 285J at /scratch1/jc_905/harxhar-clean); the scripts run THERE.
            # This repo carries neither them nor the prediction cache, so numbers are shown with their
            # reproduce-command, never recomputed locally from data that isn't present.
print("setup ok | local resid_prep cells:",
      sorted(p.name for p in Path(CACHE_ROOT).glob("*")) if Path(CACHE_ROOT).is_dir() else "(none)")

---
## 1 · Verify — the base evolution

`enetreg2` = HAR×{open,close} + cumrv×close (fixed-α single pass). Four **orthogonal** improvers, each
downstream-subsumed by the tree, **stack** to the locked linear floor `linbest`. Each base form is one
cache cell built by `prep …slim <base_kind>`; its **base-alone** QLIKE is the `base_alone_qlike=…` line
that build prints. (The full deployed stack on a base would instead be
`chunk_collect xgb_all_buckets_tw1000_<base_kind>_rf480_slim resid_subset <label>` — that's ch. 03.)

In [ ]:
def prep_cmd(base_kind, q):
    return f"{PY} resid_amortized.py prep xgb all_buckets 1000 1 480 slim {base_kind}   # base_alone_qlike={q:.5f}"
base = pd.DataFrame([
    ("plain enet",                               "enet",              0.12516, "—"),
    ("enetreg2 (HAR×{open,close}+cumrv×close)",   "enetreg2",          0.12314, "base"),
    ("+ har5rank (rank-space close HAR-damp)",   "enetreg2_har5rank", 0.12293, "−0.00021"),
    ("+ ivmag (VIX magnitude, not rank)",        "enetreg2_ivmag",    0.12301, "−0.00013"),
    ("+ sig (vol-arrival Bowley path moments)",  "enetreg2_sig",      0.12302, "−0.00012"),
    ("+ exogrel (long-window rolling-rel exog)", "enetreg2_exogrel",  0.12305, "−0.00009"),
    ("linbest (all 4 stacked)",                  "enetreg2_linbest",  0.12266, "−0.00048"),
], columns=["base form", "base_kind", "qlike", "Δ vs enetreg2 (isolated)"])
base["reproduce"] = [prep_cmd(bk, q) for bk, q in zip(base.base_kind, base.qlike)]
assert_backed(base)
assert base.qlike.min() == 0.12266  # linbest is the floor
have = [bk for bk in base.base_kind
        if (Path(CACHE_ROOT) / f"xgb_all_buckets_tw1000_{bk}_rf480_slim").is_dir()]
display(base.set_index("base form"))
print("linbest = 0.12266 — the locked linear foundation the DL residual builds on (ch. 04).")
print(f"base caches vendored locally: {have or '(none — cluster-side; rebuild with the prep command shown)'}")

**Source — how each base-alone QLIKE is computed.** The seven base forms are cluster `prep` runs (their prediction cache is not in this repo), but the local linear machinery they call — the fixed-α warm-started walk-forward elastic-net fit, its OOS replay, and the Duan-smeared QLIKE — is folded from source. These three functions emit every `base_alone_qlike=…` value above (values from the CARC base-cache runs; source shown here):

<details>
<summary><code>resid_amortized.py :: _cadence_enet</code></summary>

```python
def _cadence_enet(Xs, y, train_win, refit, alpha=0.001, l1=0.2):
    """Cadence-refit ELASTIC-NET base coef/intercept (the winning linear base, 0.12530). Warm-started
    coordinate descent chains the rolling refits (no exact rank-1 for L1; warm-start is the cheap route)."""
    from sklearn.linear_model import ElasticNet

    n = len(Xs)
    starts = list(range(train_win, n, refit))
    en = ElasticNet(alpha=alpha, l1_ratio=l1, warm_start=True, max_iter=1000, tol=1e-3)
    coefs = np.empty((len(starts), Xs.shape[1]), dtype=np.float64)
    intercepts = np.empty(len(starts), dtype=np.float64)
    for i, t_r in enumerate(starts):
        en.fit(Xs[t_r - train_win : t_r], y[t_r - train_win : t_r])
        coefs[i] = en.coef_
        intercepts[i] = float(en.intercept_)
    return np.asarray(starts, dtype=np.int64), coefs, intercepts
```

</details>

<details>
<summary><code>resid_amortized.py :: _cadence_ridge_oos</code></summary>

```python
def _cadence_ridge_oos(Xs, train_win, starts, coefs, intercepts):
    """Cadence-refit Ridge OOS preds: within block [t_r, t_{r+1}) use that block's coef
    (a matvec) — free from the cadence coefs, no slow every-bar incremental. Ridge and the
    tree then refit on the SAME cadence (internally consistent baseline)."""
    n = len(Xs)
    oos = np.empty(n - train_win, dtype=np.float64)
    for i, t_r in enumerate(starts):
        t_r = int(t_r)
        t_end = int(starts[i + 1]) if i + 1 < len(starts) else n
        oos[t_r - train_win : t_end - train_win] = (
            Xs[t_r:t_end] @ coefs[i] + intercepts[i]
        )
    return oos
```

</details>

<details>
<summary><code>resid_amortized.py :: _qlike</code></summary>

```python
def _qlike(preds, y, base, train_win):
    yo, bo = y[train_win:], base[train_win:]
    pr, tr = apply_duan_smearing(preds, yo, bo)
    m = (tr > 0) & (pr > 0)
    r = tr[m] / pr[m]
    return float(np.mean(r - np.log(r) - 1))
```

</details>

**The locked base's two engineered blocks, folded from source.** `cumrv×close` is the single biggest
linear edge (the CUMRV block below); `HAR×{open,close}` is the session-edge REGIME block. Both are exactly
the columns the FWL attribution scores.

In [ ]:
display(show_one(_cumrv_close))
display(show_one(_regime_interactions))

## 2 · Interpret — why the base is at its floor (FWL block attribution)

The whole decomposition is **one reproduce-command** — `$PY fwl_attribution.py` — whose FULL row
reproduces the 0.12314 base-alone QLIKE and whose Type-III leave-one-out gives each block's unique
contribution. The block partition it uses is folded below (`block_of`); the table is its printed output,
and the linbest "legible-not-lever" collapse is the `chunk_collect` it pairs with.

In [ ]:
fwl = pd.DataFrame([
    ("HAR",    6,   -0.02208, "96% of explainable variance (R² 0.594 of 0.617)"),
    ("EXOG",   359, -0.00449, "the whole raw-exog panel"),
    ("REGIME", 4,   -0.00225, "HAR×{open,close} session-edge interaction"),
    ("CUMRV",  1,   -0.00122, "~27× more value per feature than the 359 exog"),
], columns=["block", "nfeat", "unique ΔQLIKE (Type-III)", "note"])
fwl["reproduce"] = f"{PY} fwl_attribution.py   # FULL fit reproduces 0.12314; this IS the Type-III row"
assert_backed(fwl)
display(fwl.set_index("block"))

# linbest is a *legible* base, not a QLIKE lever: its linear edge is subsumed by the tree/EBM.
collapse = pd.DataFrame([
    ("linbest − enetreg2, base-alone (linear)", -0.00048,
        f"{PY} resid_amortized.py prep ...slim enetreg2_linbest (0.12266) vs ...slim enetreg2 (0.12314)"),
    ("linbest − enetreg2, +EBM regime stage",   -0.00002,
        f"{PY} resid_amortized.py chunk_collect xgb_all_buckets_tw1000_enetreg2_linbest_rf480_slim resid_subset <label>"
        f"   # 0.12033 on BOTH bases → the linear gain collapses"),
], columns=["comparison", "ΔQLIKE", "reproduce"])
assert_backed(collapse)
display(collapse)
display(show_one(block_of))   # the exact block partition used by fwl_attribution.py

**Source — the FWL block-attribution driver.** The block table above is the printed output of `fwl_attribution.main()` (its `block_of` partition is folded just above). The FULL row reproduces the 0.12314 base-alone QLIKE; each Type-III row drops one block and refits the same fixed-α walk-forward enet. Value from the CARC `enetreg2` base-cache run (cluster-only cache); driver source shown here:

<details>
<summary><code>fwl_attribution.py :: main</code></summary>

```python
def main() -> None:
    cell = None
    for c in sorted(glob.glob(f"{CACHE_ROOT}/*_enetreg2_rf480_slim")):
        if os.path.exists(f"{c}/feats.json") and os.path.exists(f"{c}/Xs.npy"):
            cell = c
            break
    assert cell, "no enetreg2 cell with feats.json + Xs.npy found"
    Xs = np.ascontiguousarray(np.load(f"{cell}/Xs.npy").astype(np.float64))
    y = np.load(f"{cell}/y.npy")
    base = np.load(f"{cell}/base.npy")
    feats = json.load(open(f"{cell}/feats.json"))
    assert len(feats) == Xs.shape[1], "feats/Xs width mismatch %d/%d" % (
        len(feats),
        Xs.shape[1],
    )

    blocks: dict[str, list[int]] = {b: [] for b in ORDER}
    for i, f in enumerate(feats):
        blocks[block_of(f)].append(i)
    print("FWL attribution on cell: %s" % cell, flush=True)
    print(
        "block sizes: " + ", ".join("%s=%d" % (b, len(blocks[b])) for b in ORDER),
        flush=True,
    )

    yo = y[TRAIN_WIN:]
    sst = float(np.sum((yo - yo.mean()) ** 2))

    def fit(cols):
        Xsub = np.ascontiguousarray(Xs[:, sorted(cols)])
        starts, coefs, intercepts = _cadence_enet(Xsub, y, TRAIN_WIN, REFIT)
        oos = _cadence_ridge_oos(Xsub, TRAIN_WIN, starts, coefs, intercepts)
        q = _qlike(oos, y, base, TRAIN_WIN)
        r2 = 1.0 - float(np.sum((yo - oos) ** 2)) / sst
        return q, r2, np.asarray(coefs, dtype=np.float64).mean(axis=0)

    # ---- Type I: sequential (each block added in ORDER) ----
    print("\n=== TYPE I  (sequential / incremental) ===", flush=True)
    cum: list[int] = []
    prev_q = prev_r2 = None
    for b in ORDER:
        cum += blocks[b]
        q, r2, _ = fit(cum)
        dq = "" if prev_q is None else "  dQLIKE=%+.5f" % (q - prev_q)
        dr = "" if prev_r2 is None else "  dR2=%+.5f" % (r2 - prev_r2)
        print(
            "  +%-7s nfeat=%4d  QLIKE=%.5f  R2=%+.5f%s%s"
            % (b, len(cum), q, r2, dq, dr),
            flush=True,
        )
        prev_q, prev_r2 = q, r2

    full_cols = sum(blocks.values(), [])
    q_full, r2_full, coefs_full = fit(full_cols)
    print(
        "  FULL    nfeat=%4d  QLIKE=%.5f  R2=%+.5f   (sanity: enetreg2 base-alone = 0.12314)"
        % (len(full_cols), q_full, r2_full),
        flush=True,
    )

    # ---- Type III: leave-one-out (unique contribution net of ALL others) ----
    print("\n=== TYPE III  (leave-one-out / unique) ===", flush=True)
    full_set = set(full_cols)
    for b in ORDER:
        drop = set(blocks[b])
        cols = [i for i in full_cols if i not in drop]
        q, r2, _ = fit(cols)
        print(
            "  -%-7s nfeat=%4d  QLIKE=%.5f  unique_dQLIKE=%+.5f  unique_dR2=%+.5f"
            % (b, len(cols), q, q_full - q, r2_full - r2),
            flush=True,
        )

    # ---- Partial coefficients from the full fit (FWL: coef = partial, net of the rest) ----
    print(
        "\n=== PARTIAL COEFS (full fit, mean across refits) -- REGIME + CUMRV ===",
        flush=True,
    )
    pairs = [(feats[i], coefs_full[i]) for i in (blocks["REGIME"] + blocks["CUMRV"])]
    for nm, c in sorted(pairs, key=lambda t: -abs(t[1])):
        print("  COEF %-26s %+.5f" % (nm, c), flush=True)
    print("\nFWL_ATTRIBUTION_DONE", flush=True)
```

</details>

- **`cumrv` is ~27× more valuable per feature than the raw exog** — direct measurement of the mechanism
  dominates piling on indirect series (the data-to-buy case).
- The improvers **stack to 87%** of their orthogonal sum (−0.00048 vs −0.00055) — near-additive.
- But **`linbest` is a legible base, not a QLIKE lever**: its −0.00048 linear edge **collapses to −0.00002
  at the +EBM stage** — the tree/EBM subsume it. So linbest is the *clean residual* for the DL stage, and
  the deployed 0.12035 was already at the floor. The penalty is **not basis-invariant** (unpenalizing the
  dense HAR block is neutral + legible); a global α can't price sparse close-gated columns (ch. 03).